In [ ]:
# init
from pathlib import Path
from importlib import reload
import toolsGeneral.main as tgm
import toolsGeneral.logger as tgl
import toolsPandas.helpers as tph
import toolsSync.main as tsm
import os
import boto3
from dotenv import load_dotenv
import pandas as pd
import copy
import re
import shutil

def pckgs_reload():
    reload(tgm)
    reload(tph)
    reload(tsm)

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data"
logger = tgl.initiate_logger('logger')

214


In [210]:
# init
# state files
process_state_file = Path(DATA_DIR / 'process_state.json')
process_state = tgm.load(process_state_file)
print(len(process_state))
dups_test_state_file = Path(DATA_DIR / 'dups_test_state.json')
dups_test_state = tgm.load(dups_test_state_file)
print(len(dups_test_state))
first_level_test_state_file = Path(DATA_DIR / 'first_level_test_state.json')
first_level_test_state = tgm.load(first_level_test_state_file)
print(len(first_level_test_state))

214
214
68


In [ ]:
# init
load_dotenv()
bucket_name = os.environ["B2_BUCKET_NAME"]
session = boto3.session.Session()
s3 = session.client(
    service_name="s3",
    aws_access_key_id=os.environ["B2_KEY_ID"],
    aws_secret_access_key=os.environ["B2_APPLICATION_KEY"],
    endpoint_url=os.environ["B2_ENDPOINT"]
)

config = {'root':ROOT, 's3':s3, 'logger':logger}

# read bucket

In [5]:
task_meta = {
    'scrape':{'dir':Path("data/raw/osm countries queries/")}, 
    'clean':{'dir':Path("data/cleaned/")}, 
    'test_basic':{'dir':Path("data/tests results/osm basic test")},
    'test_first_level':{'dir':Path("data/tests results/osm first level test")},
    'test_duplicates':{'dir':Path("data/tests results/osm duplicates test")}
}
task_responses = {t:s3.list_objects_v2(Bucket=bucket_name, Prefix=task_meta[t]['dir'].as_posix()) for t in task_meta.keys()}
print([len(res['Contents']) for task, res in task_responses.items()])

sync_state = copy.deepcopy(task_meta)
for task, res in task_responses.items():
    sync_state[task]['b2_countries'] = {Path(obj['Key']).parent.name for obj in res['Contents']}
    sync_state[task]['local_countries'] = {dir.name for dir in (ROOT / sync_state[task]['dir']).glob('*') if dir.is_dir()}
    sync_state[task]['b2_files'] = {obj['Key'] for obj in res['Contents']}
    sync_state[task]['local_files'] = {dir.relative_to(ROOT).as_posix() for dir in (ROOT / sync_state[task]['dir']).rglob('*') if dir.is_file()}
    b2_countries_not_in_local = tgm.complement(
        sync_state[task]['b2_countries'],
        sync_state[task]['local_countries']
    )
    b2_files_not_in_local = tgm.complement(sync_state[task]['b2_files'], sync_state[task]['local_files'])
    sync_state[task]['b2_files_not_in_local'] = b2_files_not_in_local
    local_files_not_in_b2 = tgm.complement(sync_state[task]['local_files'], sync_state[task]['b2_files'])
    sync_state[task]['local_files_not_in_b2'] = local_files_not_in_b2
    
# get len
sync_state_resume = copy.deepcopy(sync_state)
for task,val in sync_state_resume.items():
    for k,v in sync_state_resume[task].items():
        if k in ['dir']: continue
        sync_state_resume[task][k] = len(v)

[345, 85, 85, 78, 57]


In [6]:
pd.DataFrame(sync_state_resume).T

,dir,b2_countries,local_countries,b2_files,local_files,b2_files_not_in_local,local_files_not_in_b2
scrape,data\raw\osm countries queries,85,85,345,345,0,0
clean,data\cleaned,85,85,85,86,0,1
test_basic,data\tests results\osm basic test,85,85,85,86,0,1
test_first_level,data\tests results\osm first level test,68,68,78,78,0,0
test_duplicates,data\tests results\osm duplicates test,34,34,57,57,0,0


In [7]:
sync_state['test_first_level']['b2_files_not_in_local']

[]

In [10]:
countries_missing = ['Canada', 'Germany', 'France', 'Peru']

# download state files

In [ ]:
# RUN 'python scripts/pull_from_B2.py' 

In [220]:
process_state_new_file = Path(DATA_DIR / "process_state_new.json")
process_state_new = tgm.load(process_state_new_file)

In [221]:
[process_state_new[c]['test_basic'] for c in countries_missing]

[{'status': 'ok', 'last_run': '2026-01-25T19:13:28', 'error': None},
 {'status': 'ok', 'last_run': '2026-01-25T19:13:28', 'error': None},
 {'status': 'ok', 'last_run': '2026-01-25T19:13:27', 'error': None},
 {'status': 'ok', 'last_run': '2026-01-25T19:13:28', 'error': None}]

### Replace after review

In [2]:
replace_files = [
    Path(DATA_DIR / "process_state.json"), 
    Path(DATA_DIR / "first_level_test_state.json"), 
    Path(DATA_DIR / "dups_test_state.json")
]

In [3]:
for file in replace_files:
    if file.exists():
        file.unlink()
    new_file = file.with_name(file.stem + '_new.json')
    new_file.rename(file)

# download countries files

In [8]:
task = 'clean'
to_download_countries = {Path(file).parent.name for file in sync_state[task]['b2_files_not_in_local']}
to_download_countries = countries_missing
print(len(to_download_countries))
print(to_download_countries)

4
['Canada', 'Germany', 'France', 'Peru']


In [12]:
# In oder to sync them more easily, just delete the local directory
for country in to_download_countries:
    try:
        dir = (ROOT / task_meta[task]['dir'] / country)
        dir.rmdir
        print(f"Directory deleted: {dir}")
    except Exception as e:
        print(e)

Directory deleted: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\cleaned\Canada
Directory deleted: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\cleaned\Germany
Directory deleted: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\cleaned\France
Directory deleted: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\cleaned\Peru


In [13]:
tsm.donwload_country_data_from_bucket(to_download_countries, bucket_name, task_meta[task]['dir'], ROOT / task_meta[task]['dir'], s3, logger)

[2026-01-26 14:44:02] [INFO] :   * Countries to get data: 4
[2026-01-26 14:44:02] [INFO] :     * Found in B2: 4, Missing in B2: 0
[2026-01-26 14:44:02] [INFO] :   * Downloading data for countries: 4
[2026-01-26 14:44:02] [INFO] :     * Directory: 'data\cleaned' -> 'c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\cleaned'
[2026-01-26 14:44:02] [INFO] :     * (1/4) Country France files found: 1
[2026-01-26 14:44:02] [INFO] :       * Skip download of existing file France_cleaned.pkl
[2026-01-26 14:44:02] [INFO] :     * (2/4) Country Peru files found: 1
[2026-01-26 14:44:02] [INFO] :       * Skip download of existing file Peru_cleaned.pkl
[2026-01-26 14:44:02] [INFO] :     * (3/4) Country Germany files found: 1
[2026-01-26 14:44:02] [INFO] :       * Skip download of existing file Germany_cleaned.pkl
[2026-01-26 14:44:02] [INFO] :     * (4/4) Country Canada files found: 1
[2026-01-26 14:44:02] [INFO] :       * Skip download of existing file Canada_cleaned

['France', 'Peru', 'Germany', 'Canada']

In [199]:
countries_missing = ['Canada','Germany','France','Peru']
print(tgm.intersection(sync_state[task]['b2_countries'], countries_missing))
print(tgm.intersection(sync_state[task]['local_countries'], countries_missing))

['France', 'Canada', 'Germany', 'Peru']
['France', 'Canada', 'Germany', 'Peru']


# upload state files

In [ ]:
tsm.upload_file_to_backblaze(DATA_DIR / 'osmMetaCountrDict.json', config)

[2026-01-24 19:12:36] [INFO] : Uploaded c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\osmMetaCountrDict.json to Backblaze successfully


{'status': 'ok', 'status_type': None, 'data': None}

In [1]:
tsm.upload_file_to_backblaze(DATA_DIR / 'process_state.json', config)

214
214
70


[2026-01-25 23:50:59] [INFO] : Uploaded c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\process_state.json to Backblaze successfully


{'status': 'ok', 'status_type': None, 'data': None}

In [174]:
tsm.upload_file_to_backblaze(DATA_DIR / 'first_level_test_state.json', config)

[2026-01-25 17:06:41] [INFO] : Uploaded c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\first_level_test_state.json to Backblaze successfully


{'status': 'ok', 'status_type': None, 'data': None}

# upload data

In [162]:
task = 'test_duplicates'
local_countries_not_in_b2 = tgm.complement(sync_state[task]['local_countries'], sync_state[task]['b2_countries'])
len(local_countries_not_in_b2)

28

In [163]:
# countries_to_upload = tgm.intersection(['Canada','Germany','France','Peru'], local_countries_not_in_b2)
countries_to_upload = local_countries_not_in_b2
# countries_to_upload = ['Benin']
print(len(countries_to_upload))

28


In [164]:
dirs = [ROOT / task_meta[task]['dir'] / country for country in countries_to_upload]
print(len(dirs))

28


In [165]:
for dir in dirs:
    tsm.upload_dir_files_to_backblaze(dir, config)

[2026-01-25 16:45:46] [INFO] : Uploading directory: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\tests results\osm duplicates test\UnitedKingdom
[2026-01-25 16:45:46] [INFO] : Number of files found: 1
[2026-01-25 16:45:46] [INFO] : Uploading UnitedKingdom_dups_test_res_0.pkl ...
[2026-01-25 16:45:47] [INFO] : Uploaded UnitedKingdom_dups_test_res_0.pkl to Backblaze successfully
[2026-01-25 16:45:47] [INFO] : Uploading directory: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\tests results\osm duplicates test\Bangladesh
[2026-01-25 16:45:47] [INFO] : Number of files found: 1
[2026-01-25 16:45:47] [INFO] : Uploading Bangladesh_dups_test_res_0.pkl ...
[2026-01-25 16:45:48] [INFO] : Uploaded Bangladesh_dups_test_res_0.pkl to Backblaze successfully
[2026-01-25 16:45:48] [INFO] : Uploading directory: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\tests results\osm duplicates test\S